# Lang-Bridge — dub a team's videos and links

Runs the whole pipeline — **fetch → transcribe → translate → voice → mix** — on a team's
working folder: its own videos and/or links (videos, playlists, channels), from any language into any languages, with no app needed.
The Lang-Bridge app can open the results to check, fix and listen.

## Where it runs

| | Before **Run all** |
|---|---|
| **Kaggle** | Import this notebook (File → Import notebook → Link: `https://raw.githubusercontent.com/LetPastorSpeakInYourLanguage/lang_recomp/main/notebooks/lang_bridge.ipynb`); Settings → Accelerator **GPU T4 x2** (both are used), Internet **on** (phone-verified account). Use `STORAGE = bucket` to keep results. |
| **Colab** | Runtime → Change runtime type → **T4 GPU**. Drive is connected by itself when a setting points into `/content/drive/…`. |
| **Your own machine** | Jupyter with Python ≥ 3.10 (a fresh virtual environment is best). An NVIDIA GPU is used if there is one, otherwise the CPU (fine for transcripts, slow for voices). PyTorch and ffmpeg are installed if missing. |

Everything is set in the **Settings** cell, the token and bucket included.

Safe to stop at any time: run again with the same settings to continue (or see RUN_FOLDER).
Guide: [https://github.com/LetPastorSpeakInYourLanguage/lang_recomp/tree/main/notebooks](https://github.com/LetPastorSpeakInYourLanguage/lang_recomp/tree/main/notebooks)

In [ ]:
#@title Settings
WORKSPACE = ""  #@param {type:"string"}
#@markdown The team's working folder — always needed, even if it starts empty or all videos come from links. It holds the team's videos (any nesting; `name.srt` beside a video is used instead of transcribing), `lang-bridge.json` (language, targets, every link added) and all results in `.lb/`. Colab: a Drive folder, e.g. `/content/drive/MyDrive/LangBridge/Preaching`; your machine: any folder. With STORAGE = bucket: leave empty (a local copy in WORK_DIR).
STORAGE = "folder"  #@param ["folder", "bucket"]
#@markdown `folder`: the workspace is WORKSPACE. `bucket`: the workspace is BUCKET — downloaded first, results pushed back after every stage (the app syncs it). On Kaggle use `bucket`.
BUCKET = ""  #@param {type:"string"}
#@markdown A Hugging Face bucket, `namespace/name`. Needed for STORAGE = bucket; with a folder it is left alone.
LINKS = ""  #@param {type:"string"}
#@markdown Links to add: videos, playlists or channels (YouTube or any site yt-dlp supports), separated by spaces, or a `.txt` file with one link per line. They are recorded in the workspace and downloaded into it; links added before are always included.
LANGUAGE = "en"  #@param {type:"string"}
#@markdown The videos' language (e.g. `en`, `tr`, `am`). Must match an existing workspace's.
TARGETS = "am"  #@param {type:"string"}
#@markdown Languages to dub into, e.g. `am om ti`. Added to the workspace's own (both are done).
STAGES = "fetch transcribe translate voice mix"  #@param {type:"string"}
#@markdown Which stages: e.g. `fetch transcribe` for transcripts only.
DUB_LIMIT = 0  #@param {type:"integer"}
#@markdown Voice and mix only the first N videos (0 = all). The rest are transcribed and translated.
LIMIT = 0  #@param {type:"integer"}
#@markdown Work on only the first N videos (0 = all).
ASR_MODELS = "am=badrex/Ethio-ASR-amharic"  #@param {type:"string"}
#@markdown A speech recogniser per source language from Hugging Face (`lang=repo`, space separated). Languages not named use Whisper large-v3.
ALIGNERS = ""  #@param {type:"string"}
#@markdown Word aligners per language (`lang=repo`), over the built-in ones for ~27 languages.
VOICE = "native"  #@param ["native", "clone"]
#@markdown `native`: each line is first spoken by a native speaker's voice (Amharic for now), then turned into the character's voice with Seed-VC, so pauses and rhythm are the language's own. `clone`: the character's voice is cloned straight from its English lines (the old way; languages without native voices always use it).
CAPTIONS = 0  #@param {type:"integer"}
ALIGN_CAPTIONS = 0  #@param {type:"integer"}
#@markdown YouTube only: also take YouTube's captions for the first N videos, force-align the first M, and compare with Whisper.
RUN_FOLDER = ""  #@param {type:"string"}
#@markdown Continue a run: `runs/<run>` (in the workspace's `.lb/`), or a full path. Settings above are then ignored.

HF_TOKEN = ""  #@param {type:"string"}
#@markdown Your [Hugging Face token](https://huggingface.co/settings/tokens) (write access if you use a bucket). **Never share or commit this notebook with a token filled in.**
DEVICE = "auto"  #@param ["auto", "cpu", "1 GPU", "2 GPUs"]
#@markdown `auto`: every GPU there is (Kaggle T4 x2 → 2, Colab T4 → 1), else the CPU.
WORK_DIR = ""  #@param {type:"string"}
#@markdown Code, models and scratch files. Empty: `/kaggle/working`, `/content`, or `~/lang-bridge` on your own machine.
CODE_BRANCH = "main"  #@param {type:"string"}

In [ ]:
#@title Get Lang-Bridge
import importlib.util, io, os, shutil, subprocess, sys, urllib.request, zipfile
from pathlib import Path

def _runtime():
    if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
        return 'kaggle'
    try:
        if importlib.util.find_spec('google.colab'):
            return 'colab'
    except (ImportError, ValueError):
        pass
    return 'local'

RUNTIME = _runtime()
ROOT = Path(WORK_DIR).expanduser() if WORK_DIR.strip() else Path({'kaggle': '/kaggle/working', 'colab': '/content'}.get(RUNTIME) or Path.home() / 'lang-bridge')
ROOT.mkdir(parents=True, exist_ok=True)
BRANCH = CODE_BRANCH.strip() or 'main'
# On your own machine, a notebook opened from inside a Lang-Bridge checkout uses that checkout.
_here = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'worker' / 'lb_worker').is_dir() and (p / 'app').is_dir()), None)
if RUNTIME == 'local' and _here:
    CODE = _here  # as it is, on whatever branch it has checked out (CODE_BRANCH is not used)
    BRANCH = subprocess.run(['git', '-C', str(CODE), 'rev-parse', '--abbrev-ref', 'HEAD'], capture_output=True, text=True).stdout.strip() or 'checkout'
else:
    CODE = ROOT / 'lang_recomp'
    if shutil.which('git'):
        if not (CODE / '.git').exists():
            shutil.rmtree(CODE, ignore_errors=True)
            subprocess.run(['git', 'clone', '-q', '--depth', '1', '-b', BRANCH, 'https://github.com/LetPastorSpeakInYourLanguage/lang_recomp', str(CODE)], check=True)
        else:
            subprocess.run(['git', '-C', str(CODE), 'fetch', '-q', '--depth', '1', 'origin', BRANCH], check=True)
            subprocess.run(['git', '-C', str(CODE), 'checkout', '-q', '-f', '-B', BRANCH, 'FETCH_HEAD'], check=True)
    else:  # no git on this machine: the branch as a zip
        shutil.rmtree(CODE, ignore_errors=True)
        with urllib.request.urlopen('https://github.com/LetPastorSpeakInYourLanguage/lang_recomp/archive/refs/heads/' + BRANCH + '.zip') as r:
            zipfile.ZipFile(io.BytesIO(r.read())).extractall(ROOT)
        next(p for p in ROOT.glob('lang_recomp-*') if p.is_dir()).rename(CODE)
sys.path[:0] = [str(CODE), str(CODE / 'worker')]
_v = subprocess.run(['git', '-C', str(CODE), 'log', '-1', '--format=%h (%cd)'], capture_output=True, text=True).stdout.strip() if (CODE / '.git').exists() else 'zip'
print(f'Lang-Bridge {BRANCH} {_v} in {CODE} · running on {RUNTIME}')

In [ ]:
#@title Set up this machine
from lb_worker import env
M = env.setup(RUNTIME, ROOT, DEVICE, STORAGE, WORKSPACE, BUCKET, HF_TOKEN)

In [ ]:
#@title Run
from lb_worker.research import run_manifest, run_workspace

pairs = lambda text: dict(p.split('=', 1) for p in text.split() if '=' in p)
if RUN_FOLDER.strip():
    folder = RUN_FOLDER.strip() if os.path.isabs(RUN_FOLDER.strip()) else M['workspace'] / '.lb' / RUN_FOLDER.strip()
    result = run_manifest(folder, hf_token=M['token'], bucket=M['bucket'] and M['bucket'] + '/.lb')
else:
    result = run_workspace(M['workspace'], LANGUAGE, TARGETS.split(), LINKS, STAGES.split(),
                           bucket=M['bucket'] if M['storage'] == 'bucket' else None, hf_token=M['token'],
                           dub_limit=DUB_LIMIT or None, limit=LIMIT or None, captions=CAPTIONS,
                           align_captions=ALIGN_CAPTIONS, asr_models=pairs(ASR_MODELS), aligners=pairs(ALIGNERS),
                           options={**M['options'], 'voice_path': 'native_vc' if VOICE == 'native' else 'clone'})

## Look at the results

- **Without the app**: the dubbed videos are in `.lb/library/<work>/<video>/export/` in the
  workspace (the folder, or the bucket).
- **With a bucket**: in the Lang-Bridge app, Settings → Folders → add one of kind
  **Hugging Face bucket**, then **Sync**; the run's progress shows as it goes and its results
  open by themselves.
- **With a workspace folder the app can see** (Drive for Desktop, or this PC): Settings →
  Folders → add it as a **results folder**; its runs appear under Home → Runs.

Done with the GPU? Colab: **Runtime → Disconnect and delete runtime**; Kaggle: **Stop session**.